In [3]:
from dotenv import load_dotenv
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool

from app.core.config import settings

load_dotenv()

# 创建连接池
checkpoint_pool = AsyncConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预准备sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
await checkpoint_pool.open()
# 等待连接池就绪
await checkpoint_pool.wait()

# 初始化checkpointer
checkpointer = AsyncPostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
await checkpointer.setup()

In [12]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},
)

# 4.创建智能体
agent = create_agent(
    model=model,
    tools=[],
    checkpointer=checkpointer
)

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": "checkpointer-test-01"}}

# 5.调用模型
# 第一次，告知自己的信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我叫小明，今年30岁。")]},
    config=config,
)

print(result)

{'messages': [HumanMessage(content='我叫小明，今年30岁。', additional_kwargs={}, response_metadata={}, id='71df517f-cf7e-48b2-b20a-9e246d3c456e'), AIMessage(content='你好，小明！很高兴认识你。30岁正是充满活力和可能性的年纪呢。有什么我可以帮你的吗？无论是聊聊生活、工作，还是想探讨某个话题，我都在这里。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 11, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '91f62c36-1e18-489c-9bd9-1556ddafa3a4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18558-181f-76a0-be25-e14934247316-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 43, 'total_tokens': 54, 'input_token_det

In [10]:
# 第二次，询问信息
result = await agent.ainvoke(
    {"messages": [HumanMessage(content="我今年多大？")]},
    config=config,
)
print(result["messages"][-1].content)

你刚才已经告诉我：**你叫小明，今年30岁**。所以按你提供的信息，**你今年30岁**。😊

如果你是想让我根据出生日期重新计算，请告诉我你的**出生年月日**和**按哪一天算**，我可以帮你算周岁/虚岁。


In [7]:
snapshot = await agent.aget_state(config)
print(snapshot.values)

{'messages': [HumanMessage(content='我叫小明，今年30岁。', additional_kwargs={}, response_metadata={}, id='71df517f-cf7e-48b2-b20a-9e246d3c456e'), AIMessage(content='你好，小明！很高兴认识你。30岁正是充满活力和可能性的年纪呢。有什么我可以帮你的吗？无论是聊聊生活、工作，还是想探讨某个话题，我都在这里。😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 11, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 11}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '91f62c36-1e18-489c-9bd9-1556ddafa3a4', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a18558-181f-76a0-be25-e14934247316-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 43, 'total_tokens': 54, 'input_token_det

In [6]:
messages = snapshot.values.get("messages", [])

for message in messages:
    message.pretty_print()

================================ Human Message =================================

我叫小明，今年30岁。
================================== Ai Message ==================================

你好，小明！很高兴认识你。30岁正是充满活力和可能性的年纪呢。有什么我可以帮你的吗？无论是聊聊生活、工作，还是想探讨某个话题，我都在这里。😊
================================ Human Message =================================

我叫小明，今年30岁。
================================== Ai Message ==================================

你好呀，小明！又见面啦 😊 30岁的你，是刚过完生日，还是想和我再确认一下这个“设定”呢？哈哈～  
不管怎样，我都记住啦：**小明，30岁**。  
今天想聊点什么？工作、生活、兴趣，或者随便聊聊都行～
================================ Human Message =================================

我今年多大？
================================== Ai Message ==================================

你今年 **30岁** 呀，小明！😊 刚才你告诉我两次啦～ 需要我帮你记着这个“人设”吗？哈哈～


In [11]:
thread_id = config["configurable"]["thread_id"]
await checkpointer.adelete_thread(thread_id)

In [12]:
snapshot = await agent.aget_state(config)
print(snapshot.values)

{}
